# Problema del Peluquero Dormido — Análisis Formal y Verificación Experimental

**Universidad del Valle — Departamento de Ingeniería de Sistemas**  
**Sistemas Operativos — Taller Final**  

---

## Frente 1 — Reproducción Determinista del Fallo (*Lost Wakeup*)

### 1. Objetivo
El objetivo de esta sección es reproducir e ilustrar de manera completamente determinista el fallo de concurrencia conocido como **Lost Wakeup** (Despertar Perdido) en la versión incorrecta del problema del peluquero dormido, donde no se emplea una sincronización atómica para la secuencia de verificación de estado y bloqueo (dormir).

## 2. Fundamento Teórico

El fenómeno del **Lost Wakeup** (OSTEP, Capítulo 30) ocurre cuando un hilo lector/receptor se prepara para bloquearse esperando una señal, pero justo antes de efectuar la llamada de bloqueo (p. ej., `wait()`), es interrumpido por el planificador (scheduler). Otro hilo (el emisor) se ejecuta, altera la condición de espera y envía la señal de despertar (`signal` o `set`). Como el primer hilo aún no está técnicamente en estado de espera, la señal se pierde. Cuando el primer hilo reanuda su ejecución, procede a bloquearse y queda suspendido indefinidamente porque la señal ya fue emitida y no volverá a ocurrir.

En el problema del peluquero dormido:
1. El barbero comprueba `waiting == 0` (no hay clientes).
2. Justo antes de irse a dormir (ejecutar `wait()`), llega un cliente.
3. El cliente ve que el barbero está "despierto" (pues el estado aún no es "durmiendo"), por lo que no le envía la señal de despertar (`set()`), limitándose a sentarse en la silla de espera.
4. El barbero reanuda su ejecución, asume que no hay clientes porque ya hizo la evaluación previa, y se duerme (`wait()`).
5. El barbero queda durmiendo indefinidamente (Lost Wakeup) y el cliente esperando en la silla para siempre.

In [ ]:
# ===========================================================================
# FRENTE 1 — Implementación INCORRECTA del Problema del Peluquero Dormido
# Propósito: Demostrar el Lost Wakeup de forma determinista
# ===========================================================================

import threading
import time
import random

# Parámetros de la barbería
NUM_CHAIRS = 3          # Número de sillas de espera
INJECT_DELAY = 0.15     # Retardo inyectado para simular la interrupción del scheduler

# Variables de estado compartidas sin protección adecuada de exclusión mutua
waiting = 0             # Clientes esperando
barber_sleeping = False # Estado del barbero
barber_event = threading.Event()  # Evento para despertar al barbero

# Registro de eventos
start_time = time.perf_counter()
event_log = []
log_lock = threading.Lock()  # Exclusión mutua únicamente para el orden del log impreso

def log(actor, message):
    """Registra un evento con marca de tiempo precisa."""
    with log_lock:
        ts = time.perf_counter() - start_time
        entry = f"[T={ts:.4f}s] [{actor}] {message}"
        event_log.append(entry)
        print(entry)

def barber_incorrect():
    global waiting, barber_sleeping
    
    log("BARBERO", "Inicio de turno. Verificando si hay clientes...")
    
    # Verificación del estado de la sala
    # Entre esta lectura de 'waiting' y el bloqueo físico en 'wait()'
    # ocurre la interrupción del scheduler (ventana de vulnerabilidad).
    if waiting == 0:
        log("BARBERO", "No hay clientes. Me preparo para dormir...")
        
        # INYECCIÓN DEL RETARDO:
        # Forzamos al scheduler a dar paso a otro hilo (el cliente) antes de bloquearnos.
        time.sleep(INJECT_DELAY)
        
        # Transición al estado durmiendo y bloqueo
        barber_sleeping = True
        log("BARBERO", "Entrando en estado durmiendo. Ejecutando wait()...")
        
        # Usamos un timeout de 2.0 segundos para el experimento
        woke_up = barber_event.wait(timeout=2.0)
        
        if woke_up:
            barber_sleeping = False
            log("BARBERO", "Desperté. Atendiendo al cliente.")
        else:
            log("BARBERO", " FALLO: Nadie me despertó. Lost Wakeup confirmado.")
    else:
        log("BARBERO", f"Hay {waiting} cliente(s) esperando. Atendiendo...")

def client_incorrect(client_id, arrival_delay):
    global waiting, barber_sleeping
    
    # Simula el viaje del cliente a la barbería
    time.sleep(arrival_delay)
    log(f"CLIENTE-{client_id}", "Llegando a la barbería...")
    
    # Condicion para evaluar sillas libres
    if waiting < NUM_CHAIRS:
        waiting += 1
        log(f"CLIENTE-{client_id}", f"Me senté en una silla. Esperando = {waiting}")
        
        # El cliente comprueba si el barbero duerme
        # Como el barbero está en su ventana de retardo, barber_sleeping es aún False.
        # El cliente asume erróneamente que el barbero está despierto y NO envía la señal.
        if barber_sleeping:
            log(f"CLIENTE-{client_id}", "El barbero está durmiendo. Enviando señal de despertar (set)... ")
            barber_event.set()
        else:
            log(f"CLIENTE-{client_id}", "Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]")
    else:
        log(f"CLIENTE-{client_id}", "Sala llena. Me retiro.")

### 3. Ejecución Experimental del Fallo

In [ ]:
print("=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===\n")

# Limpieza e inicialización
event_log.clear()
barber_event.clear()
waiting = 0
barber_sleeping = False

# Hilos: El barbero inicia de inmediato (T=0). El cliente llega en T=0.05s, 
# cayendo exactamente en la ventana de vulnerabilidad del barbero (0s a 0.15s).
t_barber = threading.Thread(target=barber_incorrect)
t_client = threading.Thread(target=client_incorrect, args=(1, 0.05))

t_start = time.perf_counter()
t_barber.start()
t_client.start()

t_barber.join()
t_client.join()
t_duration = time.perf_counter() - t_start

print(f"\nDuración de la simulación: {t_duration:.4f} segundos")
if t_duration >= 2.0:
    print("\nRESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).")
else:
    print("\nRESULTADO: El barbero se despertó a tiempo.")

### 4. Análisis de la Intercalación Registrada
A continuación, podemos examinar la bitácora exacta de eventos para rastrear el entrelazamiento temporal (*interleaving*) que causó la pérdida de la señal de despertar:

In [ ]:
print("=== SECUENCIA DE INTERCALACIÓN REGISTRADA ===\n")
for entry in event_log:
    print(entry)

print("\nExplicación del entrelazamiento:")
print("1. El barbero inicia y verifica 'waiting == 0', lo cual es verdadero.")
print("2. El planificador suspende al barbero (simulado por INJECT_DELAY).")
print("3. El cliente llega, incrementa 'waiting' a 1, e inspecciona 'barber_sleeping'.")
print("4. Dado que el barbero aún no ha ejecutado 'wait()', 'barber_sleeping' es False.")
print("5. El cliente asume que el barbero lo atenderá y finaliza sin enviar la señal (set).")
print("6. El barbero reanuda, establece 'barber_sleeping = True' and se bloquea en 'wait()'.")
print("7. El barbero queda suspendido hasta que expira el timeout, demostrando el Lost Wakeup.")

---

## Frente 2 — Modelo e Invariantes del Sistema

### 1. Modelo Formal del Sistema

Para analizar formalmente la concurrencia en la barbería, definimos los componentes del sistema:

#### 1.1 Actores y sus Estados
- **Barbero:** DURMIENDO, VERIFICANDO, TRABAJANDO.
- **Clientes:** VIAJANDO, LLEGANDO, ESPERANDO, RECIBIENDO_CORTE, ATENDIDO, RECHAZADO.

#### 1.2 Recursos Compartidos y Variables
- **Sala de Espera:** Capacidad limitada de $N$ sillas (recurso crítico).
- `sillas_espera`: Contador de clientes esperando en la sala.
- `barbero_durmiendo`: Bandera lógica del estado del barbero.

--- 

### 2. Invariantes del Sistema (Propiedades de Seguridad)

- **Invariante 1 (Capacidad de la Sala):**
  $$0 \le sillas\_espera \le CAPACIDAD\_MAX$$
- **Invariante 2 (Ausencia de Negligencia del Barbero):**
  $$\text{Si } barbero\_durmiendo = True \implies sillas\_espera = 0$$
- **Invariante 3 (Ausencia de Espera Inútil):**
  $$\text{Si } sillas\_espera > 0 \implies barbero\_durmiendo = False$$

### 3. Implementación Simplificada de Verificación de Invariantes

In [29]:
# ===========================================================================
# FRENTE 2 — FUNCIÓN DE VERIFICACIÓN DE INVARIANTES
# ===========================================================================

def verificar_invariantes(sillas_espera, barbero_durmiendo, capacidad_max=3):
    """Valida las aserciones de seguridad del sistema."""
    # Invariante 1: Capacidad de la sala
    assert 0 <= sillas_espera <= capacidad_max, \
        f"Violación: sillas_espera ({sillas_espera}) fuera de rango [0, {capacidad_max}]."
    
    # Invariante 2: Si el barbero duerme, la sala de espera debe estar vacía
    if barbero_durmiendo:
        assert sillas_espera == 0, \
            f"Violación: El barbero duerme pero hay {sillas_espera} clientes esperando."
            
    # Invariante 3: Si hay clientes esperando, el barbero no puede estar dormido
    if sillas_espera > 0:
        assert not barbero_durmiendo, \
            "Violación: Hay clientes esperando pero el barbero está dormido."

# ===========================================================================
# PRUEBA DE FUNCIONAMIENTO DE LOS INVARIANTES
# ===========================================================================
print("=== PRUEBA DE INVARIANTES (FRENTE 2) ===\n")

# Caso 1: Estado válido
print("1. Evaluando estado válido (1 cliente esperando, barbero despierto)... ")
verificar_invariantes(sillas_espera=1, barbero_durmiendo=False)
print("   Resultado: [OK] Estado seguro.")

# Caso 2: Estado inválido (fuerza un error)
print("\n2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... ")
try:
    verificar_invariantes(sillas_espera=2, barbero_durmiendo=True)
    print("   Resultado: [ERROR] Se permitió un estado inconsistente.")
except AssertionError as e:
    print(f"   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] {e}")

=== PRUEBA DE INVARIANTES (FRENTE 2) ===

1. Evaluando estado válido (1 cliente esperando, barbero despierto)... 
   Resultado: [OK] Estado seguro.

2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... 
   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] Violación: El barbero duerme pero hay 2 clientes esperando.


---

## Frente 3 — Implementación Correcta y Robusta del Peluquero Dormido

### 1. Justificación del Mecanismo de Sincronización

Utilizaremos una única variable de condición `threading.Condition()`. Esta primitiva envuelve un lock interno que garantiza exclusión mutua para leer y escribir sobre las variables globales, además de permitir la sincronización mediante `wait()` y `notify_all()` para evitar el Lost Wakeup.

In [31]:
# ===========================================================================
# FRENTE 3 — IMPLEMENTACIÓN CORRECTA DEL PELUQUERO (Estilo Clase)
# ===========================================================================

import threading
import time
import random

# Primitiva de sincronización del Monitor
condicion = threading.Condition()

# Parámetros y variables de estado
CAPACIDAD_MAX = 2
sillas_espera = 0
cola_espera = []            # Cola FIFO para garantizar el turno y evitar despertares espurios

# Estados lógicos de sincronización
barbero_durmiendo = False
cliente_sentado_en_silla = False
corte_terminado = False

# Contadores para auditoría
total_creados = 0
total_atendidos = 0
total_rechazados = 0

def barbero():
    global sillas_espera, barbero_durmiendo, cliente_sentado_en_silla, corte_terminado, total_atendidos
    
    print("[Barbero] Abriendo la peluquería...")
    
    # El barbero atiende a un número determinado de clientes en la corrida rápida (ej. 3 clientes)
    for _ in range(3):
        with condicion:
            # Mientras no haya clientes en la cola, el barbero duerme
            while len(cola_espera) == 0:
                barbero_durmiendo = True
                print("[Barbero] No hay clientes. Me duermo en la silla...")
                verificar_invariantes(sillas_espera, barbero_durmiendo, CAPACIDAD_MAX)
                condicion.wait()
            
            # Despierta y selecciona al primer cliente de la fila
            barbero_durmiendo = False
            cliente_actual = cola_espera[0]
            print(f"[Barbero] Despierto. Llamando al cliente {cliente_actual}.")
            
            # Notifica a los hilos clientes que la silla está libre
            condicion.notify_all()
            
            # Espera a que el cliente se desplace y se siente en la silla de corte
            while not cliente_sentado_en_silla:
                condicion.wait()
        
        # Simulación del corte de cabello (FUERA del lock para permitir concurrencia en la sala)
        print(f"[Barbero] Cortando el cabello del cliente {cliente_actual}...")
        time.sleep(0.08)
        
        with condicion:
            # Termina el corte de cabello
            corte_terminado = True
            cliente_sentado_en_silla = False
            print(f"[Barbero] Corte finalizado para el cliente {cliente_actual}.")
            # Notifica al cliente en la silla que el corte terminó
            condicion.notify_all()

def cliente(id_cliente, delay_llegada):
    global sillas_espera, barbero_durmiendo, cliente_sentado_en_silla, corte_terminado, total_atendidos, total_rechazados
    
    # Simular tiempo de viaje
    time.sleep(delay_llegada)
    
    with condicion:
        print(f"  [Cliente {id_cliente}] Llegando a la barbería...")
        
        # Si la sala de espera está llena, se va inmediatamente
        if sillas_espera >= CAPACIDAD_MAX:
            print(f"  [Cliente {id_cliente}] Sala llena. Me retiro sin corte.")
            total_rechazados += 1
            return
            
        # Toma una silla de espera y se añade a la cola
        sillas_espera += 1
        cola_espera.append(id_cliente)
        print(f"  [Cliente {id_cliente}] Se sentó en la sala (esperando={sillas_espera})")
        verificar_invariantes(sillas_espera, barbero_durmiendo, CAPACIDAD_MAX)
        
        # Si el barbero está dormido, lo despierta
        if barbero_durmiendo:
            print(f"  [Cliente {id_cliente}] ¡Despertando al barbero!")
            condicion.notify_all()
            
        # Espera en la sala de espera hasta que sea su turno y la silla esté libre
        while cola_espera[0] != id_cliente or cliente_sentado_en_silla:
            condicion.wait()
            
        # Pasa a la silla de corte: libera su silla de espera y sale de la cola
        cola_espera.pop(0)
        sillas_espera -= 1
        cliente_sentado_en_silla = True
        corte_terminado = False
        print(f"  [Cliente {id_cliente}] Pasa a la silla de corte. Libera silla de espera.")
        verificar_invariantes(sillas_espera, barbero_durmiendo, CAPACIDAD_MAX)
        
        # Notifica al barbero que ya se sentó en la silla de corte
        condicion.notify_all()
        
        # Espera en la silla de corte a que el barbero termine
        while not corte_terminado:
            condicion.wait()
            
        print(f"  [Cliente {id_cliente}] Corte recibido con éxito. Saliendo de la barbería.")
        total_atendidos += 1
        verificar_invariantes(sillas_espera, barbero_durmiendo, CAPACIDAD_MAX)

# ===========================================================================
# PRUEBA DE EJECUCIÓN (FRENTE 3)
# ===========================================================================
print("=== PRUEBA DE HILOS CON MONITOR CORRECTO ===\n")

# Inicializar variables globales para la prueba
sillas_espera = 0
cola_espera = []
barbero_durmiendo = False
cliente_sentado_en_silla = False
corte_terminado = False
total_creados = 3
total_atendidos = 0
total_rechazados = 0

# Lanzamiento de hilos
t_barb = threading.Thread(target=barbero)
t_c1 = threading.Thread(target=cliente, args=(1, 0.0))
t_c2 = threading.Thread(target=cliente, args=(2, 0.01))
t_c3 = threading.Thread(target=cliente, args=(3, 0.02))

t_barb.start()
t_c1.start()
t_c2.start()
t_c3.start()

t_c1.join()
t_c2.join()
t_c3.join()
t_barb.join() # El barbero termina tras atender 3 clientes

print(f"\nResumen Final: Creados={total_creados}, Atendidos={total_atendidos}, Rechazados={total_rechazados}")
assert total_creados == total_atendidos + total_rechazados, "Error en la conservación de clientes."
print("✅ Prueba de conservación aprobada.")

Exception in thread Thread-13:
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/threading.py", line 973, in _bootstrap_inner
    self.run()
  File "/Users/solarte1999/Library/Python/3.9/lib/python/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/threading.py", line 910, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/fp/lhx130rn61zcsq461g6j0ydr0000gn/T/ipykernel_3224/841117972.py", line 85, in cliente
  File "/var/folders/fp/lhx130rn61zcsq461g6j0ydr0000gn/T/ipykernel_3224/2709407368.py", line 13, in verificar_invariantes
AssertionError: Violación: El barbero duerme pero hay 1 clientes esperando.
Exception in thread Thread-14:
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frame

=== PRUEBA DE HILOS CON MONITOR CORRECTO ===

[Barbero] Abriendo la peluquería...
[Barbero] No hay clientes. Me duermo en la silla...
  [Cliente 1] Llegando a la barbería...
  [Cliente 1] Se sentó en la sala (esperando=1)
  [Cliente 2] Llegando a la barbería...
  [Cliente 2] Se sentó en la sala (esperando=2)
  [Cliente 3] Llegando a la barbería...
  [Cliente 3] Sala llena. Me retiro sin corte.


KeyboardInterrupt: 

---

## Frente 4 — Harness de Pruebas Adversariales

### 1. Diseño de Escenarios

Para garantizar que la implementación funciona bajo cualquier carga, creamos una función para lanzar pruebas automáticas bajo diferentes configuraciones.

In [ ]:
# ===========================================================================
# FRENTE 4 — HARNESS DE PRUEBAS ADVERSARIALES Y AUDITORÍA
# ===========================================================================

def ejecutar_escenario_prueba(nombre, cap_sillas, num_clientes, delays):
    global sillas_espera, cola_espera, barbero_durmiendo, cliente_sentado_en_silla, corte_terminado
    global total_creados, total_atendidos, total_rechazados
    
    print("=" * 75)
    print(f"INICIANDO ESCENARIO: {nombre.upper()}")
    print(f"Sillas: {cap_sillas} | Clientes: {num_clientes}")
    print("=" * 75)
    
    # Reiniciar estado global
    sillas_espera = 0
    cola_espera = []
    barbero_durmiendo = False
    cliente_sentado_en_silla = False
    corte_terminado = False
    
    total_creados = num_clientes
    total_atendidos = 0
    total_rechazados = 0
    
    # Hilo del barbero: atiende exactamente al número de clientes que serán atendidos (num_clientes - rechazados esperados)
    # Para simplificar, hacemos que el barbero atiende hasta que termine el experimento.
    barber_activo = True
    
    def barbero_test():
        global sillas_espera, barbero_durmiendo, cliente_sentado_en_silla, corte_terminado
        while barber_activo or len(cola_espera) > 0:
            with condicion:
                while len(cola_espera) == 0 and barber_activo:
                    barbero_durmiendo = True
                    condicion.wait()
                if not barber_activo and len(cola_espera) == 0:
                    break
                barbero_durmiendo = False
                cliente_actual = cola_espera[0]
                cliente_sentado_en_silla = True
                corte_terminado = False
                condicion.notify_all()
                while not cliente_sentado_en_silla:
                    condicion.wait()
            time.sleep(0.01)
            with condicion:
                corte_terminado = True
                cliente_sentado_en_silla = False
                condicion.notify_all()
                
    t_barb = threading.Thread(target=barbero_test)
    t_barb.start()
    
    hilos_clientes = []
    for i in range(num_clientes):
        t_cli = threading.Thread(target=cliente, args=(i + 1, delays[i]))
        hilos_clientes.append(t_cli)
        t_cli.start()
        
    for t in hilos_clientes:
        t.join()
        
    with condicion:
        barber_activo = False
        condicion.notify_all()
    t_barb.join()
    
    # Verificación final de conservación
    assert total_creados == total_atendidos + total_rechazados, "Fallo en la vivacidad."
    print(f"\nResultado: Creados={total_creados}, Atendidos={total_atendidos}, Rechazados={total_rechazados}")
    print("✅ Todos los invariantes verificados.\n")

# Ejecutar una prueba adversarial rápida
ejecutar_escenario_prueba("Ráfaga Simultánea", cap_sillas=1, num_clientes=4, delays=[0.0]*4)